# Chapter 2: Build GPT From Scratch

Character-level GPT trained on tiny-Shakespeare. Every stage builds on the last — run cells in order.

See `../RUNBOOK.md` for the why/enterprise-notes/interview-angles that accompany this code.

## Stage 1: Data loading + character-level tokenization

In [ ]:
# Download the tiny Shakespeare dataset (~1MB)
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")
print("\nFirst 300 characters:\n")
print(text[:300])

### Build the vocabulary
Find every unique character in the dataset — that's our entire vocabulary. No subwords, no BPE merges yet (that's Chapter 4).

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"All characters: {''.join(chars)!r}")

### Encoder / decoder
`stoi` (string-to-int) and `itos` (int-to-string) are the entire tokenizer. `encode` and `decode` are just dictionary lookups.

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

test = "Hello"
encoded = encode(test)
print(f"{test!r} encoded: {encoded}")
print(f"Decoded back: {decode(encoded)!r}")

### Encode the entire dataset into a tensor

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}, dtype: {data.dtype}")
print(f"First 100 encoded characters:\n{data[:100]}")

### Train / validation split (90 / 10)
Never evaluate on data the model trained on — validation loss is our only honest signal for overfitting.

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training characters:   {len(train_data):,}")
print(f"Validation characters: {len(val_data):,}")

### Context windows (`block_size`)
The model never sees the whole dataset at once — only a fixed-size window. One window of length `block_size` actually yields `block_size` separate training examples (predict the next char at every position).

In [ ]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("One block of text yields these context -> target pairs:\n")
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"context: {decode(context.tolist())!r:20} -> target: {decode([target.item()])!r}")

### Batching
Stack many independent context windows together so the GPU processes them in parallel instead of one at a time.

In [ ]:
torch.manual_seed(1337)
batch_size = 4  # independent sequences processed in parallel

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:', xb.shape)
print(xb)
print('targets:', yb.shape)
print(yb)

**Checkpoint:** at this point you have `get_batch('train')` / `get_batch('val')` producing `(batch_size, block_size)` integer tensors ready to feed into embeddings. Next: the Bigram Language Model — the simplest possible baseline, built before self-attention so we can *feel* why it fails.

## Stage 2: The Bigram Language Model (baseline)

See `WALKTHROUGH.md` in this folder for the full line-by-line breakdown, shape tracing, and the guide-dog/blind-hiker training analogy.

This is deliberately the *dumbest possible* language model: given one character, predict the next, with zero memory of anything before it. We build and train it first so the self-attention section (next) has something concrete to improve on.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]  # only the last time step's predictions
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Initial loss: {loss.item():.4f}")
print(f"Expected loss for random guessing: {-torch.log(torch.tensor(1.0/vocab_size)):.4f}")

If `Initial loss` is close to `Expected loss`, the model is correctly initialized (i.e. currently guessing uniformly at random over the vocabulary — it hasn't learned anything yet).

In [ ]:
# Generate from the UNTRAINED model — expect pure gibberish
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=100)
print(decode(generated[0].tolist()))

### Train the Bigram model

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 1000 == 0:
        print(f"Step {steps}: loss = {loss.item():.4f}")

print(f"\nFinal loss: {loss.item():.4f}")

In [ ]:
# Generate from the TRAINED model — better, but still poor: no memory beyond one character
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))

**Checkpoint:** loss should drop from ~4.7 (random) to ~2.4-2.5. The output is *slightly* more English-shaped than noise but still gibberish — because the bigram model only ever looks at the single most recent character. It can't tell whether `h` came from `t` (predict `e`), `s` (predict `i`), or something else. That's exactly the problem self-attention solves next: letting characters look back at everything before them, not just the last one.

**Stop here and run this section before continuing** — confirm your loss curve and generated text look like the above, then we move to Part 9: the mathematical trick behind self-attention.